Nama:

NIM:

Kelas: Sistem Informasi 24


In [ ]:
import pandas as pd
import sqlite3


In [ ]:
# ==========================================
# 1. EXTRACT: Mengambil data mentah
# ==========================================
dataset_path = "/content/Train.csv"
df = pd.read_csv(dataset_path)

In [ ]:
# ==========================================
# 2. DATA CLEANING (Pembersihan Data)
# ==========================================

Print("Informasi Dataset:")
df.info()
print("\nStatistik Deskriptif:")
print(df.describe())


Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10999 entries, 0 to 10998
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   ID                   10999 non-null  int64 
 1   Warehouse_block      10999 non-null  object
 2   Mode_of_Shipment     10999 non-null  object
 3   Customer_care_calls  10999 non-null  int64 
 4   Customer_rating      10999 non-null  int64 
 5   Cost_of_the_Product  10999 non-null  int64 
 6   Prior_purchases      10999 non-null  int64 
 7   Product_importance   10999 non-null  object
 8   Gender               10999 non-null  object
 9   Discount_offered     10999 non-null  int64 
 10  Weight_in_gms        10999 non-null  int64 
 11  Reached.on.Time_Y.N  10999 non-null  int64 
dtypes: int64(8), object(4)
memory usage: 1.0+ MB

Statistik Deskriptif:
                ID  Customer_care_calls  Customer_rating  Cost_of_the_Product  \
count  10999.00000         10999

In [ ]:
print((df.isna().sum() / len(df)))

ID                     0.0
Warehouse_block        0.0
Mode_of_Shipment       0.0
Customer_care_calls    0.0
Customer_rating        0.0
Cost_of_the_Product    0.0
Prior_purchases        0.0
Product_importance     0.0
Gender                 0.0
Discount_offered       0.0
Weight_in_gms          0.0
Reached.on.Time_Y.N    0.0
dtype: float64


In [ ]:
print(f"\nJumlah duplikasi: {df.duplicated().sum()}\n")
df = df.drop_duplicates()


Jumlah duplikasi: 0



penanganan terhadap nilai yang hilang, duplikat, outliers, dan nilai yang tidak konsisten dalam dataset.

In [ ]:
results = []

cols = df.select_dtypes(include=['float64', 'int64'])

for col in cols:
  q1 = df[col].quantile(0.25)
  q3 = df[col].quantile(0.75)
  iqr = q3 - q1
  lower_bound = q1 - 1.5*iqr
  upper_bound = q3 + 1.5*iqr
  outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
  percent_outliers = (len(outliers)/len(df))*100
  results.append({'Kolom': col, 'Persentase Outliers': percent_outliers})

# Dataframe dari list hasil
results_df = pd.DataFrame(results)
results_df.set_index('Kolom', inplace=True)
results_df = results_df.rename_axis(None, axis=0).rename_axis('Kolom', axis=1)

# Tampilkan dataframe
display(results_df)

Kolom,Persentase Outliers
ID,0.000000
Customer_care_calls,0.000000
Customer_rating,0.000000
Cost_of_the_Product,0.000000
Prior_purchases,9.119011
Discount_offered,20.083644
Weight_in_gms,0.000000
Reached.on.Time_Y.N,0.000000


In [ ]:
# A. Menghapus data duplikat (jika ada)
df = df.drop_duplicates()

# B. Menghapus atau mengisi baris yang kosong (Missing Values)
df = df.dropna()

In [ ]:
# ==========================================
# 3. FEATURE ENGINEERING (Penambahan Kolom Kategori)
# ==========================================
print("3. Membuat kolom kategori untuk kebutuhan Dashboard...")

# A. Menerjemahkan Status Keterlambatan agar mudah dibaca di Dashboard
# 1 = Terlambat, 0 = Tepat Waktu
df['Status_Pengiriman'] = df['Reached.on.Time_Y.N'].map({1: 'Terlambat', 0: 'Tepat Waktu'})

# B. Membuat Kategori Berat Barang
def kelompokkan_berat(berat):
    if pd.isna(berat):
        return 'Tidak Diketahui'
    elif berat < 2000:
        return 'Ringan (< 2kg)'
    elif berat <= 4000:
        return 'Sedang (2kg - 4kg)'
    else:
        return 'Berat (> 4kg)'
df['Kategori_Berat'] = df['Weight_in_gms'].apply(kelompokkan_berat)

# C. Membuat Kategori Diskon
df['Kategori_Diskon'] = df['Discount_offered'].apply(lambda x: 'Tinggi (>10%)' if x > 10 else 'Rendah (<=10%)')

In [ ]:
# ==========================================
# 4. TRANSFORM: Memecah menjadi Star Schema (Data Warehouse)
# ==========================================
print("4. Transformasi menjadi Star Schema (Fakta & Dimensi)...")

# A. Membuat Tabel Dimensi
dim_metode = df[['Mode_of_Shipment']].drop_duplicates().reset_index(drop=True)
dim_metode['ID_Metode_Kirim'] = dim_metode.index + 1

dim_gudang = df[['Warehouse_block', 'Product_importance']].drop_duplicates().reset_index(drop=True)
dim_gudang['ID_Gudang'] = dim_gudang.index + 1

dim_pelanggan = df[['Gender', 'Customer_rating']].drop_duplicates().reset_index(drop=True)
dim_pelanggan['ID_Pelanggan'] = dim_pelanggan.index + 1

# B. Menggabungkan kunci (ID) ke dataset utama
fact_df = df.copy()
fact_df = fact_df.merge(dim_metode, on=['Mode_of_Shipment'], how='left')
fact_df = fact_df.merge(dim_gudang, on=['Warehouse_block', 'Product_importance'], how='left')
fact_df = fact_df.merge(dim_pelanggan, on=['Gender', 'Customer_rating'], how='left')

# C. Membuat Tabel Fakta
fact_pengiriman = fact_df[['ID', 'ID_Pelanggan', 'ID_Gudang', 'ID_Metode_Kirim',
                           'Cost_of_the_Product', 'Weight_in_gms', 'Kategori_Berat',
                           'Discount_offered', 'Kategori_Diskon',
                           'Customer_care_calls', 'Prior_purchases',
                           'Reached.on.Time_Y.N', 'Status_Pengiriman']]

In [ ]:
# ==========================================
# 5. LOAD: Memasukkan ke Data Warehouse (SQLite Lokal)
# ==========================================
print("5. Menyimpan tabel ke dalam file database lokal (logistik_dw.db)...")

# Membuat/koneksi ke file database SQLite
conn = sqlite3.connect('logistik_dw.db')

# Mengunggah tabel ke dalam database
print("-> Membuat Tabel Dimensi Metode Kirim...")
dim_metode.to_sql('Dim_Metode_Kirim', conn, if_exists='replace', index=False)

print("-> Membuat Tabel Dimensi Gudang Produk...")
dim_gudang.to_sql('Dim_Gudang_Produk', conn, if_exists='replace', index=False)

print("-> Membuat Tabel Dimensi Pelanggan...")
dim_pelanggan.to_sql('Dim_Pelanggan', conn, if_exists='replace', index=False)

print("-> Membuat Tabel Fakta Pengiriman...")
fact_pengiriman.to_sql('Fact_Pengiriman', conn, if_exists='replace', index=False)

# Tutup koneksi agar file tersimpan dengan sempurna
conn.close()